# Лабораторна робота №2
**Тема:** Наука про дані: підготовчий етап

## Завдання 1: Завантаження даних
Завантажити тестові структуровані файли, що містять значення VHI-індексу для кожної області. При зберіганні файлу до його імені додати дату та час завантаження. Запобігти повторному завантаженню.

In [1]:
import os
import urllib.request
import datetime
import glob

def download_noaa_data(target_dir="vhi_data"):

    os.makedirs(target_dir, exist_ok=True)
    
    now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for prov_id in range(1, 28):
        existing_files = glob.glob(f"{target_dir}/vhi_prov_{prov_id}_*.csv")
        if existing_files:
            print(f"Дані для області {prov_id} вже завантажено. Пропускаємо.")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={prov_id}&year1=1981&year2=2024&type=Mean"
        filename = f"vhi_prov_{prov_id}_{now}.csv"
        filepath = os.path.join(target_dir, filename)
        
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response:
                text = response.read().decode('utf-8')
                
            with open(filepath, 'w') as f:
                f.write(text)
            print(f"Успішно завантажено: {filename}")
            
        except Exception as e:
            print(f"Помилка завантаження для області {prov_id}: {e}")

download_noaa_data()

Успішно завантажено: vhi_prov_1_20260307_124357.csv
Успішно завантажено: vhi_prov_2_20260307_124357.csv
Успішно завантажено: vhi_prov_3_20260307_124357.csv
Успішно завантажено: vhi_prov_4_20260307_124357.csv
Успішно завантажено: vhi_prov_5_20260307_124357.csv
Успішно завантажено: vhi_prov_6_20260307_124357.csv
Успішно завантажено: vhi_prov_7_20260307_124357.csv
Успішно завантажено: vhi_prov_8_20260307_124357.csv
Успішно завантажено: vhi_prov_9_20260307_124357.csv
Успішно завантажено: vhi_prov_10_20260307_124357.csv
Успішно завантажено: vhi_prov_11_20260307_124357.csv
Успішно завантажено: vhi_prov_12_20260307_124357.csv
Успішно завантажено: vhi_prov_13_20260307_124357.csv
Успішно завантажено: vhi_prov_14_20260307_124357.csv
Успішно завантажено: vhi_prov_15_20260307_124357.csv
Успішно завантажено: vhi_prov_16_20260307_124357.csv
Успішно завантажено: vhi_prov_17_20260307_124357.csv
Успішно завантажено: vhi_prov_18_20260307_124357.csv
Успішно завантажено: vhi_prov_19_20260307_124357.csv
Ус

## Завдання 2: Зчитування та очищення даних
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, видалити зайвий текст. Замінити індекси так, щоб області індексувалася за українською абеткою.

In [2]:
import pandas as pd

def load_and_clean_data(data_dir="vhi_data"):
    data = []
    
    for file in glob.glob(f"{data_dir}/*.csv"):
        prov_id = int(os.path.basename(file).split('_')[2])
        
        with open(file, 'r') as f:
            for line in f:
                clean_line = line.replace('<tt><pre>', '').replace('</pre></tt>', '').strip()
                
                if not clean_line or 'year' in clean_line.lower():
                    continue
                    
                parts = [x.strip() for x in clean_line.split(',')]
                
                if len(parts) >= 7:
                    try:
                        vhi_val = float(parts[6])
                        if vhi_val != -1.00:
                            data.append({
                                'NOAA_ID': prov_id,
                                'Year': int(parts[0]),
                                'Week': int(parts[1]),
                                'SMN': float(parts[2]),
                                'SMT': float(parts[3]),
                                'VCI': float(parts[4]),
                                'TCI': float(parts[5]),
                                'VHI': vhi_val
                            })
                    except ValueError:
                        continue
                        
    df = pd.DataFrame(data)
    
    noaa_to_ua = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    ua_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'АР Крим',
        26: 'м. Київ', 27: 'м. Севастополь'
    }
    
    df['Province_ID'] = df['NOAA_ID'].map(noaa_to_ua)
    df['Province_Name'] = df['Province_ID'].map(ua_names)
    
    df = df.drop('NOAA_ID', axis=1)
    
    return df

df = load_and_clean_data()
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1,0.066,266.54,40.85,32.55,36.70,13,Миколаївська
1,1982,2,0.066,267.40,42.87,28.53,35.70,13,Миколаївська
2,1982,3,0.065,268.03,42.87,25.85,34.36,13,Миколаївська
3,1982,4,0.061,268.46,39.17,25.89,32.53,13,Миколаївська
4,1982,5,0.053,268.39,33.90,27.52,30.71,13,Миколаївська


## Завдання 3: Функції для вибірок
Реалізувати процедури для формування вибірок:
* Ряд VHI для області за вказаний рік;
* Ряд VHI за вказаний діапазон років для вказаних областей;
* Пошук екстремумів (min та max).

In [3]:
def get_vhi_by_year_and_province(dataframe, province_id, year):
    result = dataframe[(dataframe['Province_ID'] == province_id) & (dataframe['Year'] == year)]
    return result[['Week', 'VHI']]

def get_vhi_by_range(dataframe, province_ids, year_min, year_max):
    result = dataframe[
        (dataframe['Province_ID'].isin(province_ids)) & 
        (dataframe['Year'] >= year_min) & 
        (dataframe['Year'] <= year_max)
    ]
    return result[['Year', 'Week', 'Province_Name', 'VHI']]

def get_extremes(dataframe, province_ids, year_min, year_max):
    subset = get_vhi_by_range(dataframe, province_ids, year_min, year_max)
    
    stats = {
        'Min VHI': subset['VHI'].min(),
        'Max VHI': subset['VHI'].max(),
        'Mean VHI': subset['VHI'].mean(),
        'Median VHI': subset['VHI'].median()
    }
    return stats

## Завдання 4: Тестування функцій
Виклик створених процедур та читабельний вивід результатів.

In [4]:
print("1. Дані VHI для Вінницької області (ID 1) за 2020 рік")
vhi_2020_vinnytsia = get_vhi_by_year_and_province(df, 1, 2020)
print(vhi_2020_vinnytsia.head(), "\n")

print("2. Дані VHI для Київської (9) та Львівської (12) областей за 2018-2020 роки")
vhi_range_data = get_vhi_by_range(df, [9, 12], 2018, 2020)
print(vhi_range_data.head(), "\n")

print("3. Екстремуми для Київської (9) та Львівської (12) областей (2018-2020)")
extremes_data = get_extremes(df, [9, 12], 2018, 2020)
for key, value in extremes_data.items():
    print(f"{key}: {value:.2f}")

1. Дані VHI для Вінницької області (ID 1) за 2020 рік
       Week    VHI
47832     1  40.92
47833     2  43.19
47834     3  44.74
47835     4  45.29
47836     5  44.80 

2. Дані VHI для Київської (9) та Львівської (12) областей за 2018-2020 роки
      Year  Week Province_Name    VHI
8380  2018     1     Львівська  47.91
8381  2018     2     Львівська  48.77
8382  2018     3     Львівська  50.34
8383  2018     4     Львівська  50.10
8384  2018     5     Львівська  49.55 

3. Екстремуми для Київської (9) та Львівської (12) областей (2018-2020)
Min VHI: 20.91
Max VHI: 63.62
Mean VHI: 46.78
Median VHI: 46.84
